In [9]:
import sys
sys.path.append("../src")

import torch
from graphnet.datasets import Netzschleuder


root = "./data"
dataset_name='urban_streets'
network_name='london'
dataset = Netzschleuder(root=root, dataset_name=dataset_name, network_name=network_name)

data = dataset[0]

print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.edge_index.size(1)}")

degree = torch.bincount(data.edge_index[0])
avg_degree = degree.float().mean().item()
max_degree = degree.max().item()

print(f"\nNetwork Statistics:")
print(f"Average degree: {avg_degree:.2f}")
print(f"Maximum degree: {max_degree}")

print(f"\nFeature Information:")
print(f"Node features: {data.x.shape if hasattr(data, 'x') else 'No node features'}")

print(data)

Number of nodes: 488
Number of edges: 1458

Network Statistics:
Average degree: 2.99
Maximum degree: 5

Feature Information:
Node features: torch.Size([488, 488])
Data(x=[488, 488], edge_index=[2, 1458], num_nodes=488)


In [7]:
from graphnet.models import GCN
from torch.optim import Adam
import torch.nn.functional as F

data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[:2] = True
data.test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.test_mask[2:] = True

data.y = torch.zeros(data.num_nodes, dtype=torch.long)
data.y[0] = 1
data.y[1] = 0

model = GCN(data.num_features, 16, 2)

optimizer = Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    _, pred = model(data).max(dim=1)
    correct = pred[data.test_mask].eq(data.y[data.test_mask]).sum().item()
    acc = correct / data.test_mask.sum().item()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: -0.0250, Accuracy: 0.2486
Epoch: 001, Loss: -0.0705, Accuracy: 0.4068
Epoch: 002, Loss: -0.1114, Accuracy: 0.4972
Epoch: 003, Loss: -0.1541, Accuracy: 0.6045
Epoch: 004, Loss: -0.1994, Accuracy: 0.6271
Epoch: 005, Loss: -0.2489, Accuracy: 0.6836
Epoch: 006, Loss: -0.3020, Accuracy: 0.7288
Epoch: 007, Loss: -0.3591, Accuracy: 0.7401
Epoch: 008, Loss: -0.4197, Accuracy: 0.7853
Epoch: 009, Loss: -0.4839, Accuracy: 0.8588
Epoch: 010, Loss: -0.5518, Accuracy: 0.9209
Epoch: 011, Loss: -0.6234, Accuracy: 0.9322
Epoch: 012, Loss: -0.6989, Accuracy: 0.9435
Epoch: 013, Loss: -0.7782, Accuracy: 0.9605
Epoch: 014, Loss: -0.8616, Accuracy: 0.9718
Epoch: 015, Loss: -0.9492, Accuracy: 0.9718
Epoch: 016, Loss: -1.0410, Accuracy: 0.9718
Epoch: 017, Loss: -1.1372, Accuracy: 0.9831
Epoch: 018, Loss: -1.2379, Accuracy: 0.9831
Epoch: 019, Loss: -1.3432, Accuracy: 0.9887
Epoch: 020, Loss: -1.4530, Accuracy: 0.9887
Epoch: 021, Loss: -1.5677, Accuracy: 0.9887
Epoch: 022, Loss: -1.6872, Accur